## TP2 : Quadratic Assignment Problem

Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
np.random.seed(32)

#### Part 1 : 29 september 2025

Our problem is the Quadratic Assignment Problem (QAP). It is a combinatorial optimization problem where we have to assign a set of facilities to a set of locations in such a way that the total cost is minimized. The cost is determined by the distances between locations and the flow between facilities.

**Cities and distances**

We will generate two problems of size 10 and 100.

In [ ]:
# First, we initiate 12 cities with coordinates in a 100x100 grid.
# np.random.seed(42)

def generate_cities_random(num_cities,grid_size):
    """
    Generate num_cities in a grid of size grid_size x grid_size

    Parameters:
    num_cities (int) : the number of cities we want to generate
    grid_size (int) : the size of the grid

    Returns:
    coordinates (np.array(num_cities,2)) : Coordinates of the cities
    """ 
    pass


def plot_cities(cities, facilities):
    """
    Generate random city coordinates within a grid.

    Parameters:
    cities (np.array(num_cities, 2): Coordinates of the cities.
    
    Returns:
    A scatter plot of the cities with their indices.
    """
    plt.figure(figsize=(6, 6))
    
    # Plot all cities
    plt.scatter(cities[:, 0], cities[:, 1], color='blue', s=50, label='City')
    
    # Annotate cities
    for i, (x, y) in enumerate(cities):
        plt.text(x - 3, y - 1, f'City {i}', fontsize=12, ha='right', color='blue')
    
    plt.scatter(
        cities[facilities, 0], cities[facilities, 1],
        facecolors='none', edgecolors='black', s=200, linewidths=2,
        label='Facility'
    )
    # Annotate facilities
    for j, idx in enumerate(facilities):
        x, y = cities[idx]
        plt.text(x + 1, y + 1, f'F{j}', fontsize=12, color='black', weight='bold')

    plt.xlim(0, 100)
    plt.ylim(0, 100)
    plt.title('City Locations')
    plt.grid(True)
    plt.legend()
    plt.show()


In [ ]:
def fitness(assignment, weights_matrix , distance_matrix):
    """
    Compute the fitness of an assignement of facilities to locations.

    Parameters:
    assignement (list of int): Permutation of size num_cities
    weights_matrix (np.array[num_cities][num_cities]) : Matrix of weights
    distance_matrix (np.array[num_cities][num_cities]) : Matrix of distances
    
    Returns:
    fitness (float) : The fitness of the assignement
    """
    pass

In [ ]:
def generate_weights_random(num_cities, max_weight=25):
    """
    Generate a random weight matrix for the cities.

    Parameters:
    num_cities (int): Number of cities.
    max_weight (int): Maximum weight value.

    Returns:

    weights (np.array(num_cities, num_cities)): Weight matrix.
    """
    pass


def compute_distance_matrix(cities):
    """
    Compute the distance matrix between cities.

    Parameters:
    cities (np.array(num_cities, 2)): Coordinates of the cities.

    Returns:
    distance_matrix (np.array(num_cities, num_cities)): Distance matrix.
    """
    pass

### Greedy easy method 

In [ ]:
def greedy_easy(weights, distance_matrix):
    """
    Greedy heuristic for the Quadratic Assignment Problem:
    facilities with highest total weight -> locations with lowest total distance.
    """
    n = weights.shape[0]

    # Sum of the weights (decreasing order)
    w_sum = np.array(np.sum(weights, axis=1))
    w_sort = (np.argsort(-w_sum))


    # Sum of distances for each city (ascending order)
    d_sum = np.sum(distance_matrix, axis=1)
    d_sort = np.argsort(d_sum)
    

    assignment = np.zeros(n, dtype=int)
    for i in range(n):
        assignment[w_sort[i]] = d_sort[i]

    fitness_greedy = fitness(assignment, weights, distance_matrix)

    return assignment.tolist(), fitness_greedy

### Greedy solution (other method)

In [ ]:
def get_best_neighbour(city1, neighbour_dict, free_list):
    # Returns the best free location at the minimum distance from city1
    for city2 in neighbour_dict[city1]:
        if free_list[city2]:
            return city2
    return None

def get_two_good_places(emp1_list, emp2_list, free_list):
    # Returns two free locations that are at minimal distance from each other
    for e1, e2 in zip(emp1_list, emp2_list):
        if e1 != e2 and free_list[e1] and free_list[e2]:
            return e1, e2
    return None, None

def greedy_solution(weights, distance_matrix):
    n = weights.shape[0]
    assignment = {}

    # Distances sorted (only pairs where i < j are considered)
    idx_distance = np.argsort(distance_matrix, axis=None)
    rows_d, cols_d = np.unravel_index(idx_distance, distance_matrix.shape)

    best_neighbour = {}
    for i, j in zip(rows_d, cols_d):
        if i != j:
            best_neighbour.setdefault(i, []).append(j)

    # Sort the flows in decreasing order (strongest connections first)
    # (Sort the weights of the W matrix)
    idx_weights = np.argsort(weights, axis=None)[::-1]
    rows_w, cols_w = np.unravel_index(idx_weights, weights.shape)
    
    covered = np.zeros(n, dtype=bool)
    free_pos = np.ones(n, dtype=bool)

    # Browse the links in decreasing order of weight
    for c1, c2 in zip(rows_w, cols_w):
        if c1 == c2:
            continue
        # If a link involves two cities that have not been placed yet
        if not covered[c1] and not covered[c2]:
            # Find the best positions for both cities (minimum distance)
            e1, e2 = get_two_good_places(rows_d, cols_d, free_pos)
            if e1 is None:
                break
            assignment[c1], assignment[c2] = e1, e2
            covered[c1], covered[c2] = True, True
            free_pos[e1], free_pos[e2] = False, False
        # If only one of the two cities has already been placed
        elif not covered[c1]:
            # Find the best possible neighbor location for the already placed city (minimum distance)
            e1 = get_best_neighbour(assignment.get(c2, c2), best_neighbour, free_pos)
            if e1 is not None:
                assignment[c1] = e1
                covered[c1] = True
                free_pos[e1] = False

        elif not covered[c2]:
            # Same as above but for the second city
            e2 = get_best_neighbour(assignment.get(c1, c1), best_neighbour, free_pos)
            if e2 is not None:
                assignment[c2] = e2
                covered[c2] = True
                free_pos[e2] = False

    return assignment, fitness(assignment, weights, distance_matrix)


Can you try to understand how the two greedy algorithms work ?

- Greedy easy

- Greedy_solution

### Setting of a problem for 12 cities

In [ ]:
positions = np.array([[ 25.00616036,   5.37921308],
       [ 49.37825796,  16.97220412],
       [ 75.12183688,  28.52093225],
       [100.        ,  39.08588089],
       [ 13.01467739,  33.39197468],
       [ 37.35707173,  44.41047962],
       [ 62.58758625,  55.73908367],
       [ 87.02189107,  66.60010608],
       [  0.        ,  60.87168438],
       [ 24.86845828,  71.56597651],
       [ 50.62776159,  83.13474908],
       [ 75.23837967,  94.54979682]], dtype=float)

D = np.array([
    [0,1,2,3,1,2,3,4,2,3,4,5],
    [1,0,1,2,2,1,2,3,3,2,3,4],
    [2,1,0,1,3,2,1,2,4,3,2,3],
    [3,2,1,0,4,3,2,1,5,4,3,2],
    [1,2,3,4,0,1,2,3,1,2,3,4],
    [2,1,2,3,1,0,1,2,2,1,2,3],
    [3,2,1,2,2,1,0,1,3,2,1,2],
    [4,3,2,1,3,2,1,0,4,3,2,1],
    [2,3,4,5,1,2,3,4,0,1,2,3],
    [3,2,3,4,2,1,2,3,1,0,1,2],
    [4,3,2,3,3,2,1,2,2,1,0,1],
    [5,4,3,2,4,3,2,1,3,2,1,0]
], dtype=int)


W = np.array([[ 0,  5,  2,  4,  1,  0,  0,  6,  2,  1,  1,  1],
        [ 5,  0,  3,  0,  2,  2,  2,  0,  4,  5,  0,  0],
        [ 2,  3,  0,  0,  0,  0,  0,  5,  5,  2,  2,  2],
        [ 4,  0,  0,  0,  5,  2,  2, 10,  0,  0,  5,  5],
        [ 1,  2,  0,  5,  0, 10,  0,  0,  0,  5,  1,  1],
        [ 0,  2,  0,  2, 10,  0,  5,  1,  1,  5,  4,  0],
        [ 0,  2,  0,  2,  0,  5,  0, 10,  5,  2,  3,  3],
        [ 6,  0,  5, 10,  0,  1, 10,  0,  0,  0,  5,  0],
        [ 2,  4,  5,  0,  0,  1,  5,  0,  0,  0, 10, 10],
        [ 1,  5,  2,  0,  5,  5,  2,  0,  0,  0,  5,  0],
        [ 1,  0,  2,  5,  1,  4,  3,  5, 10,  5,  0,  2],
        [ 1,  0,  2,  5,  1,  0,  3,  0, 10,  0,  2,  0]])



## Compare the performance of the two greedy algorithms :

- First, generate a problem with random weights for 12 cities on a 100x100 grid. Generate a thousand permutations of facilities and compute the average mean value of the fitness of such permutations. Compare the value with the two greedy algorithm. How much better do they perform ?

- Now, same task with the weights just above. Plot the repartition of the cities and the assignement of the facilities for the two greedy algorithms.

#### Here we generate a random problem for 12 cities

In [ ]:
cities_positions_12 = generate_cities_random(12,100)
distances_12 = compute_distance_matrix(cities_positions_12)
weights_12 = generate_weights_random(12,25)

fitness_random_12 = # to do
fitness_greedy_easy_12 = # to do
fitness_greedy_solution_12 = # to do
print('Relative distance random fitness to easy greedy fitness for random : {}'.format(# to do))
print('Relative distance random fitness to solution_greedy for random : {}'.format(# to do))

#### Here we compute the fitness of the greedy algorithms and compare it to the random permutationsfor the given problem

In [ ]:
fitness_greedy_easy_problem = # to do
fitness_greedy_solution_problem = # to do
print('Relative distance random fitness to easy greedy fitness for random : {}'.format(# to do))
print('Relative distance random fitness to solution_greedy for random : {}'.format(# to do))

## What do you observe ? 

### Optionnal

Try to think of a better way of generating cities and weights.

What could you do to try to have something better when generating a problem ?

In [ ]:
def generate_cities_better(num_cities, grid_size=100):
    """
    Generate better city coordinates within a grid.

    Parameters:
    num_cities (int): Number of cities to generate.
    grid_size (int): Size of the grid (grid_size x grid_size).

    Returns:
    cities_coordinates : Array of shape (num_cities, 2) with city coordinates.
    """
    pass


def generate_weights_better(num_cities, max_weight=25):
    """
    Generate a better weight matrix for the cities.

    Parameters:
    num_cities (int): Number of cities.
    max_weight (int): Maximum weight value.

    Returns:

    weights (np.array(num_cities, num_cities)): Weight matrix.
    """
    pass

In [ ]:
cities_12_better = generate_cities_better(12,100)
cities_distances_12_better = compute_distance_matrix(cities_12_better)
weights_12_better = generate_weights_better(12,25)
assignement_12_easy_better, fitness_easy_greedy_12_better = # to do 
fitness_solution_greedy_12_better = # to do 
print('Relative distance random fitness to easy greedy fitness for random : {}'.format(# to do ))
print('Relative distance random fitness to solution_greedy for random : {}'.format(# to do ))

In [ ]:
#  plot_cities(cities_positions_12,assignement_12_easy_better )

### Part 2: So now we can try to implement the tabu search

We are going to test the tabu search for the given problem and see if :
- the performance is better than the greedy algorithms ?
- how many iterations are required to get a better result ?
- test the stability of the solutions when we add some noise ?

In [ ]:
# ================================================================
# Tabu Search with optional Diversification
# ================================================================
# Skeleton for implementing the Tabu Search algorithm
# to solve the Quadratic Assignment Problem (QAP).
# Fill in each function with your own logic.
# ================================================================

def swap(permutation, i, j):
    """
    Return a new permutation where the elements at indices i and j are swapped.

    Parameters:
    permutation (list or np.ndarray): Current assignment.
    i, j (int): Indices to swap.

    Returns:
    (list or np.ndarray): New permutation with i and j exchanged.
    """
    # create a copy or modify in place to swap positions i and j
    pass


def delta_i_j(permutation, i, j, weights, distance):
    """
    Compute the change in objective value if we swap facilities i and j.

    Parameters:
    permutation (list or np.ndarray): Current assignment.
    i, j (int): Indices of facilities to swap.
    weights (np.ndarray): Flow matrix.
    distance (np.ndarray): Distance matrix.

    Returns:
    float: Change in cost after swapping i and j.
    """
    # compute the fitness difference directly or with the O(n) delta formula
    pass


def best_swap(weights, distance, permutation, current_fitness, best_fitness, tabu_matrix, itr):
    """
    Identify the best neighbor of the current solution
    while respecting Tabu Search rules.
    
    Parameters:
    weights (np.ndarray) : Flow matrix of the QAP.
    distance (np.ndarray) : Distance matrix of the QAP.
    permutation (list or np.ndarray) : Current assignment of facilities to locations.
    current_fitness (float) : Cost of the current solution.
    best_fitness (float) : Best cost found so far (global best).
    tabu_matrix (np.ndarray) : Matrix that stores, for each possible swap, the iteration number until which it is tabu.
    itr (int) : Current iteration index.

    Returns:
    i, j (int): Indices of the selected swap.
    delta (float): Change in cost for the swap.
    is_best (bool): True if this move improves the global best solution.
    """
    # iterate over all pairs (i, j)
    # compute delta for each swap
    # skip tabu moves unless aspiration criterion is met
    # keep track of the best candidate
    pass


def tabu_search(weights, distance, tabu_tenure, tmax, diversification, u):
    """
    Main Tabu Search routine.
    Parameters:
    weights (np.ndarray): Flow between locations.
    distance (np.ndarray): Distance between locations.
    tabu_tenure (int): Number of iterations a move remains tabu.
    tmax (int): Total number of iterations.
    diversification (bool): Enable diversification strategy.
    u (int): Number of iterations after which diversification triggers.

    Returns:        
    best_fitness (float): Best objective value found.
    best_permutation (list or np.ndarray): Best assignment found.
    fitness_history (list): Cost of each visited solution.
    best_history (list): Best cost at each iteration.
    """
    # initialize permutation and tabu matrix
    # initialize diversification structures if enabled

    # repeat for tmax iterations:
    #   choose move (best swap or diversification)
    #   apply move and update current fitness
    #   update best solution if improved
    #   update tabu matrix
    #   update diversification bookkeeping if enabled

    # return final best solution and histories
    pass

# ------------------------------------------------
# Diversification:
# implement a mechanism that forces rarely-used
# swaps to be tried after u iterations
# ------------------------------------------------


## What is the size of a neighbourhood ?

## Optionnal : can you think of en efficient way of implementing the diversification ?


Plot the evolution of the best fitness and of the best optimal fitness for :
- the random problem you generated
- the given problem with positions, W and D
- if you want for your better problem

You can also try to change the value of the parameters.
Compare the best fitness obtained with the fitness of the greedy algorithms.